# Stock Deepseeker - 交互式回测分析

这个Notebook用于交互式分析回测结果、优化参数、可视化性能。

## 功能
1. 加载和分析回测结果
2. 性能指标计算和可视化
3. 交易分析和模式识别
4. 参数优化和敏感性分析
5. 策略对比
6. Alpha因子分析

In [ ]:
# 导入必要的库
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime, timedelta

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ 环境初始化完成")

## 1. 加载回测结果

In [ ]:
# 选择要分析的结果目录
RESULT_DIR = "institutional_results"  # 可选: backtest_results, institutional_results

# 加载数据
equity_file = Path(RESULT_DIR) / "equity_curve.csv"
trades_file = Path(RESULT_DIR) / "trades.csv"
metrics_file = Path(RESULT_DIR) / "metrics.json"

# 读取数据
equity_df = pd.read_csv(equity_file, parse_dates=['date'])
trades_df = pd.read_csv(trades_file, parse_dates=['timestamp'])

# 读取指标
with open(metrics_file, 'r') as f:
    metrics = json.load(f)

print(f"📊 数据加载完成")
print(f"   权益数据: {len(equity_df)} 条")
print(f"   交易记录: {len(trades_df)} 笔")
print(f"\n📈 关键指标:")
for key, value in metrics.items():
    if isinstance(value, float):
        if 'return' in key or 'rate' in key:
            print(f"   {key}: {value:.2%}")
        elif 'ratio' in key:
            print(f"   {key}: {value:.2f}")
        else:
            print(f"   {key}: {value:,.2f}")
    else:
        print(f"   {key}: {value}")

## 2. 权益曲线可视化

In [ ]:
# 绘制权益曲线
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# 子图1: 权益曲线
axes[0].plot(equity_df['date'], equity_df['equity'], linewidth=2, label='总权益')
axes[0].plot(equity_df['date'], equity_df['cash'], linewidth=1.5, alpha=0.7, label='现金')
axes[0].plot(equity_df['date'], equity_df['positions'], linewidth=1.5, alpha=0.7, label='持仓价值')
axes[0].set_title('权益曲线', fontsize=14, fontweight='bold')
axes[0].set_ylabel('权益 ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 子图2: 回撤
equity_values = equity_df['equity'].values
running_max = np.maximum.accumulate(equity_values)
drawdown = (equity_values - running_max) / running_max * 100

axes[1].fill_between(equity_df['date'], drawdown, 0, alpha=0.3, color='red')
axes[1].plot(equity_df['date'], drawdown, color='red', linewidth=1)
axes[1].set_title('回撤曲线', fontsize=14, fontweight='bold')
axes[1].set_ylabel('回撤 (%)')
axes[1].grid(True, alpha=0.3)

# 子图3: 每日收益率
daily_returns = equity_df['equity'].pct_change() * 100
axes[2].bar(equity_df['date'], daily_returns, alpha=0.6, 
           color=['green' if x > 0 else 'red' for x in daily_returns])
axes[2].set_title('每日收益率', fontsize=14, fontweight='bold')
axes[2].set_xlabel('日期')
axes[2].set_ylabel('收益率 (%)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✅ 权益曲线图已生成")

## 3. 交易分析

In [ ]:
# 分离买入和卖出交易
buy_trades = trades_df[trades_df['action'] == 'BUY']
sell_trades = trades_df[trades_df['action'] == 'SELL']

print(f"📊 交易统计")
print(f"   总交易: {len(trades_df)} 笔")
print(f"   买入: {len(buy_trades)} 笔")
print(f"   卖出: {len(sell_trades)} 笔")

# 盈亏分析
if 'pnl' in sell_trades.columns:
    profitable_trades = sell_trades[sell_trades['pnl'] > 0]
    losing_trades = sell_trades[sell_trades['pnl'] < 0]
    
    total_profit = profitable_trades['pnl'].sum()
    total_loss = abs(losing_trades['pnl'].sum())
    
    print(f"\n💰 盈亏分析")
    print(f"   盈利交易: {len(profitable_trades)} 笔")
    print(f"   亏损交易: {len(losing_trades)} 笔")
    print(f"   胜率: {len(profitable_trades)/len(sell_trades):.2%}")
    print(f"   总盈利: ${total_profit:,.2f}")
    print(f"   总亏损: ${total_loss:,.2f}")
    print(f"   盈亏比: {total_profit/total_loss:.2f}")
    print(f"   平均盈利: ${profitable_trades['pnl'].mean():,.2f}")
    print(f"   平均亏损: ${losing_trades['pnl'].mean():,.2f}")
    
    # 最佳和最差交易
    best_trade = sell_trades.loc[sell_trades['pnl'].idxmax()]
    worst_trade = sell_trades.loc[sell_trades['pnl'].idxmin()]
    
    print(f"\n🏆 最佳交易")
    print(f"   股票: {best_trade['symbol']}")
    print(f"   盈利: ${best_trade['pnl']:,.2f}")
    print(f"   日期: {best_trade['timestamp']}")
    
    print(f"\n💔 最差交易")
    print(f"   股票: {worst_trade['symbol']}")
    print(f"   亏损: ${worst_trade['pnl']:,.2f}")
    print(f"   日期: {worst_trade['timestamp']}")

## 4. 股票表现分析

In [ ]:
# 按股票统计
if 'pnl' in sell_trades.columns:
    symbol_stats = sell_trades.groupby('symbol')['pnl'].agg([
        ('交易次数', 'count'),
        ('总盈亏', 'sum'),
        ('平均盈亏', 'mean'),
        ('最大盈利', 'max'),
        ('最大亏损', 'min')
    ]).sort_values('总盈亏', ascending=False)
    
    print("\n📊 股票表现排行（Top 10）")
    print(symbol_stats.head(10))
    
    # 可视化Top 10股票
    top_10 = symbol_stats.head(10)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 总盈亏
    axes[0].barh(range(len(top_10)), top_10['总盈亏'].values, 
                color=['green' if x > 0 else 'red' for x in top_10['总盈亏'].values])
    axes[0].set_yticks(range(len(top_10)))
    axes[0].set_yticklabels(top_10.index)
    axes[0].set_xlabel('总盈亏 ($)')
    axes[0].set_title('Top 10 股票 - 总盈亏', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # 交易次数
    axes[1].barh(range(len(top_10)), top_10['交易次数'].values, color='steelblue')
    axes[1].set_yticks(range(len(top_10)))
    axes[1].set_yticklabels(top_10.index)
    axes[1].set_xlabel('交易次数')
    axes[1].set_title('Top 10 股票 - 交易次数', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. 月度收益分析

In [ ]:
# 计算月度收益
equity_df['date'] = pd.to_datetime(equity_df['date'])
equity_df.set_index('date', inplace=True)

monthly_returns = equity_df['equity'].resample('M').last().pct_change() * 100
monthly_returns = monthly_returns.dropna()

# 创建年-月透视表
monthly_returns_df = pd.DataFrame({
    '年': monthly_returns.index.year,
    '月': monthly_returns.index.month,
    '收益率': monthly_returns.values
})

pivot_table = monthly_returns_df.pivot_table(
    values='收益率',
    index='年',
    columns='月',
    aggfunc='first'
)

# 热力图
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
           cbar_kws={'label': '月度收益率 (%)'})
plt.title('月度收益率热力图', fontsize=14, fontweight='bold')
plt.xlabel('月份')
plt.ylabel('年份')
plt.tight_layout()
plt.show()

print(f"\n📊 月度收益统计")
print(f"   平均月收益: {monthly_returns.mean():.2f}%")
print(f"   月收益标准差: {monthly_returns.std():.2f}%")
print(f"   最佳月份: {monthly_returns.max():.2f}%")
print(f"   最差月份: {monthly_returns.min():.2f}%")
print(f"   盈利月份占比: {(monthly_returns > 0).sum() / len(monthly_returns):.2%}")

## 6. 风险指标分析

In [ ]:
# 计算各种风险指标
equity_df.reset_index(inplace=True)
returns = equity_df['equity'].pct_change().dropna()

# VaR (95%和99%)
var_95 = np.percentile(returns, 5) * 100
var_99 = np.percentile(returns, 1) * 100

# CVaR (Expected Shortfall)
cvar_95 = returns[returns <= np.percentile(returns, 5)].mean() * 100
cvar_99 = returns[returns <= np.percentile(returns, 1)].mean() * 100

# Sortino Ratio
downside_returns = returns[returns < 0]
downside_std = downside_returns.std()
sortino_ratio = (returns.mean() / downside_std * np.sqrt(252)) if downside_std > 0 else 0

# Calmar Ratio
annual_return = metrics.get('annualized_return', 0)
max_dd = abs(metrics.get('max_drawdown', 1))
calmar_ratio = annual_return / max_dd if max_dd > 0 else 0

print("\n🛡️  风险指标")
print(f"   VaR (95%): {var_95:.2f}%")
print(f"   VaR (99%): {var_99:.2f}%")
print(f"   CVaR (95%): {cvar_95:.2f}%")
print(f"   CVaR (99%): {cvar_99:.2f}%")
print(f"   夏普比率: {metrics.get('sharpe_ratio', 0):.2f}")
print(f"   索提诺比率: {sortino_ratio:.2f}")
print(f"   卡玛比率: {calmar_ratio:.2f}")
print(f"   最大回撤: {max_dd:.2%}")

# 收益分布图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 直方图
axes[0].hist(returns * 100, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(returns.mean() * 100, color='red', linestyle='--', label=f'均值: {returns.mean()*100:.2f}%')
axes[0].axvline(var_95, color='orange', linestyle='--', label=f'VaR 95%: {var_95:.2f}%')
axes[0].set_xlabel('日收益率 (%)')
axes[0].set_ylabel('频数')
axes[0].set_title('收益率分布', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Q-Q图
from scipy import stats
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q图（正态性检验）', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 市场状态分析

In [ ]:
# 如果trades_df包含regime信息
if 'regime' in trades_df.columns:
    regime_stats = trades_df.groupby('regime').agg({
        'symbol': 'count',
        'pnl': ['sum', 'mean'] if 'pnl' in trades_df.columns else 'count'
    })
    
    print("\n📊 市场状态统计")
    print(regime_stats)
    
    # 可视化
    if 'pnl' in trades_df.columns:
        regime_pnl = trades_df.groupby('regime')['pnl'].sum()
        
        plt.figure(figsize=(12, 6))
        regime_pnl.plot(kind='bar', color='steelblue')
        plt.title('不同市场状态下的总盈亏', fontweight='bold')
        plt.xlabel('市场状态')
        plt.ylabel('总盈亏 ($)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("\n⚠️  交易数据中未包含市场状态信息")

## 8. 自定义分析

在这里添加您的自定义分析代码

In [ ]:
# 您的自定义分析
print("\n🔧 在此添加自定义分析...")